In [ ]:
import os
import pandas as pd
from openai import OpenAI

# 1. Configuración: El token es público, revócalo después de esto
os.environ["HF_TOKEN"] = ""

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"]
) # Kit de herramientas de OPEN AI

In [37]:
import os
import pandas as pd
import kagglehub

# Descargar la ultima versión
dataset_path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")
FILE_PATH = dataset_path

print("Path to dataset files:", FILE_PATH)

# 2. Listar archivos para identificar el nombre exacto
dataset_files = os.listdir(FILE_PATH) # Lista de Python con los nombres de todos los elementos contenidos.
print("Archivos en el dataset:", dataset_files)

#
csv_file_name = 'training.1600000.processed.noemoticon.csv'
full_csv_path = os.path.join(FILE_PATH, csv_file_name) # concatena una carpeta y un nombre de archivo para crear una ruta completa.

# Cargar archivo en CSV
# Using 'latin-1' encoding as it's common for these types of datasets.
df = pd.read_csv(full_csv_path, encoding='latin-1') # Debe traducir los bytes (0 y 1) del archivo a letras y símbolos que tú puedas leer.

print(f"Successfully loaded '{csv_file_name}' into DataFrame 'df'.")

Using Colab cache for faster access to the 'sentiment-analysis-dataset' dataset.
Path to dataset files: /kaggle/input/sentiment-analysis-dataset
Archivos en el dataset: ['training.1600000.processed.noemoticon.csv', 'train.csv', 'testdata.manual.2009.06.14.csv', 'test.csv']
Successfully loaded 'training.1600000.processed.noemoticon.csv' into DataFrame 'df'.


In [38]:
df.head()

,polarity of tweet,id of the tweet,date of the tweet,query,user,text of the tweet
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [50]:
# Descarga y ruta
path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")
full_csv_path = os.path.join(path, 'training.1600000.processed.noemoticon.csv')

# 2. Carga del Dataset
col_names = ['polarity', 'id', 'date', 'query', 'user', 'text']
df = pd.read_csv(full_csv_path, names=col_names, encoding='latin-1')

# 3. Función de Inferencia con Qwen
def analizar_sentimiento_qwen(texto):
    if not texto or pd.isna(texto): return "Error"
    try:
        response = client.chat.completions.create(
            model="Qwen/Qwen2.5-7B-Instruct",
            messages=[
                {"role": "system", "content": "Analiza el sentimiento. Responde solo con el numero: 0 para negativo, 4 para positivo."},
                {"role": "user", "content": texto}
            ],
            max_tokens=2,
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return "Error"

# 4. Procesamiento de una Muestra
n_muestras = 20

# Antes tenías texto suelto que causaba un 'SyntaxError'
sample_df = df.sample(n_muestras, random_state=42).copy() # Extraer únicamente el texto
print(f"Iniciando análisis de {n_muestras} filas con identificador MCCLP...")

#  Usamos MCCLP_polarity ---
sample_df['MCCLP_polarity'] = sample_df['text'].apply(analizar_sentimiento_qwen) # crear una nueva columna.

# 5. Mostrar y Guardar Resultados
print("\n--- COMPARATIVA: REAL (Polarity) vs MCCLP (IA) ---")
print(sample_df[['text', 'polarity', 'MCCLP_polarity']].head(10))

# Exportar a CSV
sample_df.to_csv('resultados_MCCLP_sentiment.csv', index=False)
print("\nArchivo 'resultados_MCCLP_sentiment.csv' guardado con éxito.")

Using Colab cache for faster access to the 'sentiment-analysis-dataset' dataset.


/tmp/ipykernel_19835/3818748072.py:7: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_csv_path, names=col_names, encoding='latin-1')


Iniciando análisis de 20 filas con identificador MCCLP...

--- COMPARATIVA: REAL (Polarity) vs MCCLP (IA) ---
                                                     text polarity  \
851067  @dotboom for the record, I don't handle the Tw...        4   
982860  @heidimontag NY is great.  I live in NY! haha,...        4   
689315           portobello road and wimbledon for a bbq         0   
301605  Ok so, the paramedics just came to my house, o...        0   
760081                     i'm sick as a dog today, man.         0   
863595  @thisisryanross http://twitpic.com/4e48i sketc...        4   
431096                       seriously though ik bevries         0   
414369  @AlaskaCook Hey there! I kiiiiinda hate you ri...        0   
718044                         last night out in chicago         0   
762190  wants to know why every time I'm home alone wi...        0   

       MCCLP_polarity  
851067              4  
982860              4  
689315              4  
301605              2  